In [0]:
import uuid
from datetime import datetime, timedelta

from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, IntegerType, ArrayType
)
from pyspark.sql.functions import (
    col, lit, current_timestamp,
    arrays_zip, explode, to_timestamp
)

BRONZE_TABLE = "bronze_weather"
SILVER_TABLE = "silver_weather"
EXECUTION_LOG_TABLE = "pipeline_execution_logs"
ERROR_LOG_TABLE = "pipeline_error_logs"

RAW_BASE_PATH = (
    "/Volumes/workspace/default/products/"
    "european_climate/raw"
)

BACKFILL_CITY = "Berlin"
BACKFILL_START_DATE = "1995-01-01"
BACKFILL_END_DATE = "1995-12-31"
LOAD_TYPE = "BACKFILL"

WEATHER_COLUMNS = [
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
]

CITIES = [
    {"city": "Oslo", "country": "Norway", "region": "Northern Europe",
     "latitude": 59.9139, "longitude": 10.7522},
    {"city": "Stockholm", "country": "Sweden", "region": "Northern Europe",
     "latitude": 59.3293, "longitude": 18.0686},
    {"city": "Helsinki", "country": "Finland", "region": "Northern Europe",
     "latitude": 60.1699, "longitude": 24.9384},
    {"city": "Copenhagen", "country": "Denmark", "region": "Northern Europe",
     "latitude": 55.6761, "longitude": 12.5683},
    {"city": "London", "country": "United Kingdom", "region": "Western Europe",
     "latitude": 51.5074, "longitude": -0.1278},
    {"city": "Berlin", "country": "Germany", "region": "Central Europe",
     "latitude": 52.5200, "longitude": 13.4050},
    {"city": "Madrid", "country": "Spain", "region": "Southern Europe",
     "latitude": 40.4168, "longitude": -3.7038},
    {"city": "Rome", "country": "Italy", "region": "Southern Europe",
     "latitude": 41.9028, "longitude": 12.4964},
    {"city": "Warsaw", "country": "Poland", "region": "Eastern Europe",
     "latitude": 52.2297, "longitude": 21.0122},
]

required_tables = [
    BRONZE_TABLE,
    SILVER_TABLE,
    EXECUTION_LOG_TABLE,
    ERROR_LOG_TABLE,
]

for table_name in required_tables:
    assert spark.catalog.tableExists(table_name), f"Missing required table: {table_name}"

city_info = next(
    (c for c in CITIES if c["city"].lower() == BACKFILL_CITY.lower()),
    None
)

assert city_info is not None, f"Unknown city: {BACKFILL_CITY}"

start_dt = datetime.strptime(BACKFILL_START_DATE, "%Y-%m-%d")
end_dt = datetime.strptime(BACKFILL_END_DATE, "%Y-%m-%d")

assert start_dt <= end_dt, "BACKFILL_START_DATE must be <= BACKFILL_END_DATE"

print("Backfill city:", city_info["city"])
print("Backfill range:", BACKFILL_START_DATE, "→", BACKFILL_END_DATE)
print("Load type:", LOAD_TYPE)


Backfill city: Berlin
Backfill range: 1995-01-01 → 1995-12-31
Load type: BACKFILL


In [0]:
hourly_units_schema = StructType([
    StructField("time", StringType(), True),
    *[StructField(name, StringType(), True) for name in WEATHER_COLUMNS],
])

hourly_schema = StructType([
    StructField("time", ArrayType(StringType()), True),
    StructField("temperature_2m", ArrayType(DoubleType()), True),
    StructField("relative_humidity_2m", ArrayType(DoubleType()), True),
    StructField("apparent_temperature", ArrayType(DoubleType()), True),
    StructField("precipitation", ArrayType(DoubleType()), True),
    StructField("rain", ArrayType(DoubleType()), True),
    StructField("snowfall", ArrayType(DoubleType()), True),
    StructField("weather_code", ArrayType(IntegerType()), True),
    StructField("pressure_msl", ArrayType(DoubleType()), True),
    StructField("surface_pressure", ArrayType(DoubleType()), True),
    StructField("cloud_cover", ArrayType(DoubleType()), True),
    StructField("wind_speed_10m", ArrayType(DoubleType()), True),
    StructField("wind_direction_10m", ArrayType(DoubleType()), True),
])

raw_weather_schema = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("generationtime_ms", DoubleType(), True),
    StructField("utc_offset_seconds", IntegerType(), True),
    StructField("timezone", StringType(), True),
    StructField("timezone_abbreviation", StringType(), True),
    StructField("elevation", DoubleType(), True),
    StructField("hourly_units", hourly_units_schema, True),
    StructField("hourly", hourly_schema, True),
])

def raw_file_exists(file_path):
    try:
        dbutils.fs.head(file_path, 1)
        return True
    except Exception:
        return False

def find_raw_year_file(city_info, year):
    safe_city = city_info["city"].lower().replace(" ", "_")
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"
    filename = f"{safe_city}_{start_date}_{end_date}.json"

    candidates = [
        f"{RAW_BASE_PATH}/full/{filename}",
        f"{RAW_BASE_PATH}/incremental/{filename}",
    ]

    for path in candidates:
        if raw_file_exists(path):
            return path

    raise FileNotFoundError(
        f"No persisted raw file found for {city_info['city']} {year}. "
        "Normal backfill does not call the API."
    )

def make_backfill_batch_id(city_info, year):
    safe_city = city_info["city"].lower().replace(" ", "_")
    return f"backfill_{safe_city}_{year}-01-01_{year}-12-31"

def raw_to_bronze_backfill(city_info, file_path, year):
    batch_id = make_backfill_batch_id(city_info, year)

    raw_df = (
        spark.read
        .schema(raw_weather_schema)
        .option("multiLine", "true")
        .json(file_path)
    )

    return (
        raw_df
        .withColumn("city", lit(city_info["city"]))
        .withColumn("country", lit(city_info["country"]))
        .withColumn("region", lit(city_info["region"]))
        .withColumn("source", lit("Open-Meteo ERA5"))
        .withColumn("batch_id", lit(batch_id))
        .withColumn("load_type", lit(LOAD_TYPE))
        .withColumn("raw_file_path", lit(file_path))
        .withColumn("load_timestamp", current_timestamp())
    )

def bronze_to_silver(bronze_df):
    exploded_df = bronze_df.select(
        "city",
        "country",
        "region",
        "source",
        "batch_id",
        "load_type",
        "raw_file_path",
        col("load_timestamp").alias("bronze_load_timestamp"),
        explode(
            arrays_zip(
                col("hourly.time"),
                col("hourly.temperature_2m"),
                col("hourly.relative_humidity_2m"),
                col("hourly.apparent_temperature"),
                col("hourly.precipitation"),
                col("hourly.rain"),
                col("hourly.snowfall"),
                col("hourly.weather_code"),
                col("hourly.pressure_msl"),
                col("hourly.surface_pressure"),
                col("hourly.cloud_cover"),
                col("hourly.wind_speed_10m"),
                col("hourly.wind_direction_10m"),
            )
        ).alias("weather"),
    )

    return exploded_df.select(
        "city",
        "country",
        "region",
        to_timestamp(col("weather.time"), "yyyy-MM-dd'T'HH:mm").alias("timestamp"),
        col("weather.temperature_2m").cast("double").alias("temperature_2m"),
        col("weather.relative_humidity_2m").cast("double").alias("relative_humidity_2m"),
        col("weather.apparent_temperature").cast("double").alias("apparent_temperature"),
        col("weather.precipitation").cast("double").alias("precipitation"),
        col("weather.rain").cast("double").alias("rain"),
        col("weather.snowfall").cast("double").alias("snowfall"),
        col("weather.weather_code").cast("int").alias("weather_code"),
        col("weather.pressure_msl").cast("double").alias("pressure_msl"),
        col("weather.surface_pressure").cast("double").alias("surface_pressure"),
        col("weather.cloud_cover").cast("double").alias("cloud_cover"),
        col("weather.wind_speed_10m").cast("double").alias("wind_speed_10m"),
        col("weather.wind_direction_10m").cast("double").alias("wind_direction_10m"),
        "source",
        "batch_id",
        "load_type",
        "raw_file_path",
        "bronze_load_timestamp",
        current_timestamp().alias("load_timestamp"),
    )


In [0]:
def write_execution_log(
    run_id,
    batch_id,
    layer,
    parameter_or_file,
    start_time,
    end_time,
    status,
    rows_inserted=0,
    rows_updated=0,
    error_message=None
):
    record = [{
        "run_id": run_id,
        "batch_id": batch_id,
        "layer": layer,
        "load_type": LOAD_TYPE,
        "parameter_or_file": parameter_or_file,
        "start_time": start_time,
        "end_time": end_time,
        "status": status,
        "rows_inserted": int(rows_inserted),
        "rows_updated": int(rows_updated),
        "error_message": error_message,
    }]

    (
        spark.createDataFrame(
            record,
            schema=spark.table(EXECUTION_LOG_TABLE).schema
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(EXECUTION_LOG_TABLE)
    )

def write_error_log(run_id, batch_id, layer, source_file, error):
    record = [{
        "error_id": str(uuid.uuid4()),
        "run_id": run_id,
        "batch_id": batch_id,
        "layer": layer,
        "source_file": source_file,
        "error_type": type(error).__name__,
        "error_message": str(error),
        "error_timestamp": datetime.now(),
    }]

    (
        spark.createDataFrame(
            record,
            schema=spark.table(ERROR_LOG_TABLE).schema
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(ERROR_LOG_TABLE)
    )

def merge_bronze(bronze_df):
    target = DeltaTable.forName(spark, BRONZE_TABLE)

    (
        target.alias("target")
        .merge(
            bronze_df.alias("source"),
            "target.batch_id = source.batch_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    metrics = (
        target.history(1)
        .select("operationMetrics")
        .first()["operationMetrics"]
    )

    return (
        int(metrics.get("numTargetRowsInserted", 0)),
        int(metrics.get("numTargetRowsUpdated", 0))
    )

def merge_silver(silver_df):
    staged_df = silver_df.dropDuplicates(["city", "timestamp"])
    target = DeltaTable.forName(spark, SILVER_TABLE)

    change_condition = "NOT (" + " AND ".join(
        [f"target.{column} <=> source.{column}" for column in WEATHER_COLUMNS]
    ) + ")"

    update_values = {
        column: f"source.{column}"
        for column in WEATHER_COLUMNS
    }

    update_values.update({
        "country": "source.country",
        "region": "source.region",
        "source": "source.source",
        "batch_id": "source.batch_id",
        "load_type": "source.load_type",
        "raw_file_path": "source.raw_file_path",
        "bronze_load_timestamp": "source.bronze_load_timestamp",
        "load_timestamp": "current_timestamp()",
    })

    (
        target.alias("target")
        .merge(
            staged_df.alias("source"),
            "target.city = source.city AND target.timestamp = source.timestamp"
        )
        .whenMatchedUpdate(
            condition=change_condition,
            set=update_values
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    metrics = (
        target.history(1)
        .select("operationMetrics")
        .first()["operationMetrics"]
    )

    return (
        int(metrics.get("numTargetRowsInserted", 0)),
        int(metrics.get("numTargetRowsUpdated", 0))
    )

def get_existing_bronze(city_name, year):
    year_pattern = f"_{year}-01-01_{year}-12-31"

    candidates = (
        spark.table(BRONZE_TABLE)
        .filter(col("city") == city_name)
        .filter(col("batch_id").contains(year_pattern))
        .orderBy(col("load_timestamp").desc())
    )

    if candidates.limit(1).count() == 0:
        return None

    full_candidate = candidates.filter(col("load_type") == "FULL")
    if full_candidate.limit(1).count() > 0:
        return full_candidate.limit(1)

    return candidates.limit(1)


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

PRODUCTION_SILVER = "silver_weather"
TEST_SILVER_TABLE = "silver_backfill_demo"

# Create a separate 1995 Berlin Silver table
test_source = (
    spark.table(PRODUCTION_SILVER)
    .filter(col("city") == "Berlin")
    .filter(
        (col("timestamp") >= "1995-01-01") &
        (col("timestamp") < "1996-01-01")
    )
)

assert test_source.count() == 8760

(
    test_source.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TEST_SILVER_TABLE)
)

before = spark.table(TEST_SILVER_TABLE).count()

print("Before deletion:", before)

# Delete January 1995 from the DEMO table only
demo_target = DeltaTable.forName(
    spark,
    TEST_SILVER_TABLE
)

demo_target.delete("""
    city = 'Berlin'
    AND timestamp >= '1995-01-01'
    AND timestamp < '1995-02-01'
""")

after_delete = spark.table(TEST_SILVER_TABLE).count()

print("After deletion:", after_delete)
print("Missing records:", before - after_delete)

assert before == 8760
assert after_delete == 8016
assert before - after_delete == 744

# Redirect the backfill Silver MERGE to the demo
SILVER_TABLE = TEST_SILVER_TABLE

assert SILVER_TABLE != PRODUCTION_SILVER

print("Backfill target:", SILVER_TABLE)
print("CONTROLLED DELETION TEST PASSED")

Before deletion: 8760
After deletion: 8016
Missing records: 744
Backfill target: silver_backfill_demo
CONTROLLED DELETION TEST PASSED


In [0]:
def run_backfill(city_info, start_date, end_date):
    requested_start = datetime.strptime(start_date, "%Y-%m-%d")
    requested_end = datetime.strptime(end_date, "%Y-%m-%d")
    results = []

    for year in range(requested_start.year, requested_end.year + 1):
        segment_start = max(requested_start, datetime(year, 1, 1))
        segment_end = min(requested_end, datetime(year, 12, 31))
        segment_end_exclusive = segment_end + timedelta(days=1)

        run_id = str(uuid.uuid4())
        backfill_batch_id = make_backfill_batch_id(city_info, year)
        parameter_text = (
            f"{city_info['city']}|"
            f"{segment_start.strftime('%Y-%m-%d')}|"
            f"{segment_end.strftime('%Y-%m-%d')}"
        )

        print("\n======================================")
        print("BACKFILL:", parameter_text)

        try:
            bronze_source = get_existing_bronze(city_info["city"], year)

            if bronze_source is not None:
                source_type = "BRONZE_REUSED"
                raw_file_path = (
                    bronze_source
                    .select("raw_file_path")
                    .first()["raw_file_path"]
                )

                print("SOURCE   → EXISTING BRONZE REUSED")

            else:
                source_type = "RAW_REBUILT"
                raw_file_path = find_raw_year_file(city_info, year)

                print("SOURCE   → RAW FILE")
                print(raw_file_path)

                bronze_start = datetime.now()

                bronze_source = raw_to_bronze_backfill(
                    city_info,
                    raw_file_path,
                    year
                )

                bronze_inserted, bronze_updated = merge_bronze(
                    bronze_source
                )

                bronze_end = datetime.now()

                write_execution_log(
                    run_id,
                    backfill_batch_id,
                    "Raw-to-Bronze",
                    raw_file_path,
                    bronze_start,
                    bronze_end,
                    "SUCCESS",
                    bronze_inserted,
                    bronze_updated
                )

                print(
                    "BRONZE   → SUCCESS",
                    "| inserted:", bronze_inserted,
                    "| updated:", bronze_updated
                )

            silver_start = datetime.now()

            silver_stage = (
                bronze_to_silver(bronze_source)
                .filter(
                    (col("timestamp") >= segment_start.strftime("%Y-%m-%d"))
                    & (col("timestamp") < segment_end_exclusive.strftime("%Y-%m-%d"))
                )
                .dropDuplicates(["city", "timestamp"])
            )

            staged_rows = silver_stage.count()

            silver_inserted, silver_updated = merge_silver(
                silver_stage
            )

            silver_end = datetime.now()

            write_execution_log(
                run_id,
                backfill_batch_id,
                "Bronze-to-Silver",
                parameter_text,
                silver_start,
                silver_end,
                "SUCCESS",
                silver_inserted,
                silver_updated
            )

            print("SOURCE   →", source_type)
            print("STAGED   →", staged_rows, "rows")
            print(
                "SILVER   → SUCCESS",
                "| inserted:", silver_inserted,
                "| updated:", silver_updated
            )

            results.append({
                "year": year,
                "status": "SUCCESS",
                "source_type": source_type,
                "staged_rows": staged_rows,
                "silver_inserted": silver_inserted,
                "silver_updated": silver_updated,
            })

        except Exception as e:
            failure_time = datetime.now()

            write_execution_log(
                run_id,
                backfill_batch_id,
                "BACKFILL",
                parameter_text,
                failure_time,
                failure_time,
                "FAILURE",
                0,
                0,
                str(e)
            )

            write_error_log(
                run_id,
                backfill_batch_id,
                "BACKFILL",
                parameter_text,
                e
            )

            print("BACKFILL → FAILURE")
            print("Reason:", str(e))

            results.append({
                "year": year,
                "status": "FAILURE",
                "error": str(e),
            })

    return results

backfill_results = run_backfill(
    city_info=city_info,
    start_date=BACKFILL_START_DATE,
    end_date=BACKFILL_END_DATE
)



BACKFILL: Berlin|1995-01-01|1995-12-31
SOURCE   → EXISTING BRONZE REUSED
SOURCE   → BRONZE_REUSED
STAGED   → 8760 rows
SILVER   → SUCCESS | inserted: 744 | updated: 0


In [0]:
from pyspark.sql.functions import col

demo = spark.table("silver_backfill_demo")
production = spark.table("silver_weather")

demo_rows = demo.count()

production_rows = production.count()

demo_duplicates = (
    demo_rows
    - demo.select("city", "timestamp").distinct().count()
)

print("===== BACKFILL RECOVERY SUMMARY =====")
print("Restored demo rows:", demo_rows)
print("Demo duplicates:", demo_duplicates)
print("Production Silver rows:", production_rows)

assert demo_rows == 8760
assert demo_duplicates == 0
assert production_rows == 3550176

print("BACKFILL RECOVERY TEST PASSED")
print("PRODUCTION SILVER ROW COUNT PRESERVED")

===== BACKFILL RECOVERY SUMMARY =====
Restored demo rows: 8760
Demo duplicates: 0
Production Silver rows: 3550176
BACKFILL RECOVERY TEST PASSED
PRODUCTION SILVER ROW COUNT PRESERVED


In [0]:
verification_start = datetime.strptime(BACKFILL_START_DATE, "%Y-%m-%d")
verification_end_exclusive = (
    datetime.strptime(BACKFILL_END_DATE, "%Y-%m-%d")
    + timedelta(days=1)
)

scope = (
    spark.table(SILVER_TABLE)
    .filter(col("city") == city_info["city"])
    .filter(
        (col("timestamp") >= verification_start.strftime("%Y-%m-%d"))
        & (col("timestamp") < verification_end_exclusive.strftime("%Y-%m-%d"))
    )
)

rows_after = scope.count()
distinct_after = (
    scope
    .select("city", "timestamp")
    .distinct()
    .count()
)

duplicates_after = rows_after - distinct_after

expected_hours = int(
    (
        verification_end_exclusive
        - verification_start
    ).total_seconds()
    / 3600
)

null_timestamps = (
    scope
    .filter(col("timestamp").isNull())
    .count()
)

null_load_timestamps = (
    scope
    .filter(col("load_timestamp").isNull())
    .count()
)

print("===== BACKFILL VALIDATION =====")
print("City:", city_info["city"])
print("Range:", BACKFILL_START_DATE, "→", BACKFILL_END_DATE)
print("Rows after backfill:", rows_after)
print("Expected hourly rows:", expected_hours)
print("Distinct city + timestamp:", distinct_after)
print("Duplicates:", duplicates_after)
print("Null timestamps:", null_timestamps)
print("Null load_timestamp:", null_load_timestamps)

checks = {
    "Expected row count restored": rows_after == expected_hours,
    "No duplicate city + timestamp": duplicates_after == 0,
    "No null timestamps": null_timestamps == 0,
    "No null load_timestamp": null_load_timestamps == 0,
    "All backfill segments succeeded": all(
        item["status"] == "SUCCESS"
        for item in backfill_results
    ),
}

print("\n===== CHECKLIST =====")
for name, passed in checks.items():
    print(name, "→", "PASS" if passed else "FAIL")

assert all(checks.values()), "Backfill validation failed"

print("\nBACKFILL VALIDATION PASSED")


===== BACKFILL VALIDATION =====
City: Berlin
Range: 1995-01-01 → 1995-12-31
Rows after backfill: 8760
Expected hourly rows: 8760
Distinct city + timestamp: 8760
Duplicates: 0
Null timestamps: 0
Null load_timestamp: 0

===== CHECKLIST =====
Expected row count restored → PASS
No duplicate city + timestamp → PASS
No null timestamps → PASS
No null load_timestamp → PASS
All backfill segments succeeded → PASS

BACKFILL VALIDATION PASSED
